# Prior Topics for Incremental Updates

Using the 20-newsgroups dataset, we show how providing known labels for clusters can
reduce LLM calls for incremental updates. We use 99% of the dataset for the initial
Toponymy labeling, and then incrementally add the remaining 1% of the dataset on a
subsequent run. This subtle update should not cause major changes to topic names, so
many clusters can reuse the topic names from the initial run.

This example starts the same way as the basic usage example: load the embedded
20-newsgroups data, extract the vectors, configure an embedding model, configure an
LLM wrapper, and fit a `Toponymy` model.

In [1]:
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

For the dataset we'll use the same embedded 20-newsgroups parquet file from the
basic usage example.

In [2]:
newsgroups_df = pd.read_parquet(
    "hf://datasets/lmcinnes/20newsgroups_embedded/data/train-00000-of-00001.parquet"
)

In [3]:
newsgroups_df.head()

,post,newsgroup,embedding,map
0,\n\nI am sure some bashers of Pens fans are pr...,rec.sport.hockey,"[-0.04380008950829506, 0.08495834469795227, -0...","[-0.13199903070926666, 10.1972017288208]"
1,My brother is in the market for a high-perform...,comp.sys.ibm.pc.hardware,"[0.006855607498437166, -0.05531690642237663, -...","[11.03041934967041, 9.509867668151855]"
2,\n\n\n\n\tFinally you said what you dream abou...,talk.politics.mideast,"[0.01537406351417303, 0.03572937101125717, -0....","[1.7360589504241943, -0.31686803698539734]"
3,\nThink!\n\nIt's the SCSI card doing the DMA t...,comp.sys.ibm.pc.hardware,"[0.010156078264117241, -0.07253803312778473, -...","[10.975887298583984, 10.715202331542969]"
4,1) I have an old Jasmine drive which I cann...,comp.sys.mac.hardware,"[-0.008448092266917229, 0.06011670082807541, 0...","[10.498811721801758, 11.010639190673828]"


Extract the text, embedding vectors, and clusterable 2D vectors in the format
Toponymy expects.

In [4]:
text = newsgroups_df["post"].str.strip().values
embedding_vectors = np.stack(newsgroups_df["embedding"].values)
clusterable_vectors = np.stack(newsgroups_df["map"].values)

We will simulate an incremental update by using the first 99% of the data for the
first fit. The second fit uses the full dataset, which is equivalent to appending
the remaining documents to a previously labeled corpus.

In [5]:
initial_size = int(0.99 * len(text))
remaining_indices = np.arange(initial_size, len(text))

initial_text = text[:initial_size]
initial_embedding_vectors = embedding_vectors[:initial_size]
initial_clusterable_vectors = clusterable_vectors[:initial_size]

initial_size, len(remaining_indices)

(17988, 182)

As in the basic usage example, we need a text embedding model for semantic
similarity work inside Toponymy.

In [6]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("paraphrase-MiniLM-L3-v2")

Toponymy supports several LLM providers. This example follows the basic usage
notebook and uses a Cohere model, but you can substitute any
Toponymy LLM wrapper.

In [7]:
import sys
import warnings
from pathlib import Path

cohere_api_key = open("cohere_key.txt").read().strip()

llm = AsyncCohereNamer(
    api_key=cohere_api_key,
    model="command-r-08-2024",
    max_concurrent_requests=10  # Adjust based on rate limits
)

We'll use the same model configuration for both runs. This creates a
fresh `ToponymyClusterer` for each fit so that the second run builds cluster layers
for the full, updated dataset.

In [8]:
keyphrase_builder = KeyphraseBuilder(
    ngram_range=(1, 6),
    max_features=15_000,
    verbose=True,
)

def make_topic_model(previous_cluster_layers=None):
    return Toponymy(
        llm_wrapper=llm,
        text_embedding_model=embedding_model,
        clusterer=ToponymyClusterer(min_clusters=4, verbose=True),
        keyphrase_builder=keyphrase_builder,
        object_description="newsgroup posts",
        corpus_description="20-newsgroups dataset",
        exemplar_delimiters=["<EXAMPLE_POST>\n", "\n</EXAMPLE_POST>\n\n"],
        previous_cluster_layers=previous_cluster_layers,
    )

First fit the topic model on the initial 99% of the corpus. This is the same fit
workflow as the basic usage example, but using the subset of data we selected above.

In [9]:
initial_topic_model = make_topic_model()

In [10]:
%%time
initial_topic_model.fit(
    initial_text,
    embedding_vectors=initial_embedding_vectors,
    clusterable_vectors=initial_clusterable_vectors,
)

In [11]:
initial_topic_model.topic_names_[-1]

['Sports', 'Religious Debates', 'Vehicle Enthusiasts', 'Hardware', 'X-Windows']

Now fit a new topic model on the complete dataset. Toponymy will compare clusters
in each new layer to the prior layers from the initial run. The prior label arrays
are shorter than the current dataset, which is fine for append-only updates: Toponymy
treats them as labels for the prefix of the current vectors. When a new cluster is
sufficiently similar to a prior cluster, it reuses the previous topic name instead
of sending that prompt to the LLM.

In [13]:
updated_topic_model = make_topic_model(
    previous_cluster_layers=initial_topic_model.cluster_layers_
)

In [14]:
%%time
updated_topic_model.fit(
    text,
    embedding_vectors=embedding_vectors,
    clusterable_vectors=clusterable_vectors,
)

The updated model contains topic labels for all documents, including the 1% that
was held back from the initial fit.

In [15]:
topics_per_document = [
    cluster_layer.topic_name_vector
    for cluster_layer in updated_topic_model.cluster_layers_
]

topics_per_document[-1][remaining_indices]

array(['Hardware', 'Hardware', 'Unlabelled', 'Unlabelled',
       'Religious Debates', 'Religious Debates', 'Unlabelled',
       'Unlabelled', 'X-Windows', 'Unlabelled', 'Religious Debates',
       'Hardware', 'Religious Debates', 'Religious Debates',
       'Vehicle Enthusiasts', 'X-Windows', 'Religious Debates',
       'Vehicle Enthusiasts', 'Unlabelled', 'X-Windows', 'Sports',
       'Hardware', 'Religious Debates', 'Religious Debates',
       'Vehicle Enthusiasts', 'Religious Debates', 'Religious Debates',
       'Religious Debates', 'Unlabelled', 'Religious Debates', 'Hardware',
       'Unlabelled', 'Sports', 'Religious Debates', 'Religious Debates',
       'Hardware', 'Religious Debates', 'Unlabelled', 'Religious Debates',
       'Unlabelled', 'Religious Debates', 'Sports', 'Religious Debates',
       'Religious Debates', 'Sports', 'Religious Debates', 'Hardware',
       'Vehicle Enthusiasts', 'X-Windows', 'Unlabelled',
       'Religious Debates', 'Religious Debates', 'Vehicle En

We can also inspect how many topic names were reused in each layer during the
incremental update.

In [16]:
reused_topic_counts = [
    len(cluster_layer._prior_topic_reuse_indices)
    for cluster_layer in updated_topic_model.cluster_layers_
]

total_topic_counts = [
    len(cluster_layer.topic_names)
    for cluster_layer in updated_topic_model.cluster_layers_
]

pd.DataFrame(
    {
        "layer": np.arange(len(updated_topic_model.cluster_layers_)),
        "reused_topics": reused_topic_counts,
        "total_topics": total_topic_counts,
        "llm_named_topics": (
            np.array(total_topic_counts) - np.array(reused_topic_counts)
        ),
    }
)

,layer,reused_topics,total_topics,llm_named_topics
0,0,412,424,12
1,1,133,135,2
2,2,40,41,1
3,3,14,14,0
4,4,5,5,0


Finally, compare the highest-level topic names from the initial and updated runs.
For a small incremental update, most of these names should remain stable.

In [17]:
pd.DataFrame(
    {
        "initial_top_level_topics": pd.Series(initial_topic_model.topic_names_[-1]),
        "updated_top_level_topics": pd.Series(updated_topic_model.topic_names_[-1]),
    }
)

,initial_top_level_topics,updated_top_level_topics
0,Sports,Sports
1,Religious Debates,Religious Debates
2,Vehicle Enthusiasts,Vehicle Enthusiasts
3,Hardware,Hardware
4,X-Windows,X-Windows
